In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from umap import UMAP
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("库加载完成")

/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


库加载完成


In [2]:
# 加载数据
psth_path = '/media/ubuntu/sda/TrippleN/customize/aggregate_response/all_subjects_psth.npy'
unit_info_path = '/media/ubuntu/sda/TrippleN/customize/aggregate_response/all_subjects_unit_info.pkl'

psth_data = np.load(psth_path, mmap_mode='r')
unit_info = pd.read_pickle(unit_info_path)

print(f"PSTH数据形状: {psth_data.shape}")
print(f"单元信息记录数: {len(unit_info)}")
print(f"\n单元信息列名: {unit_info.columns.tolist()}")

PSTH数据形状: (15652, 1072, 450)
单元信息记录数: 15652

单元信息列名: ['B_SI', 'F_SI', 'O_SI', 'UnitType', 'best_r_time1', 'best_r_time2', 'pos', 'reliability_basic', 'reliability_best', 'reliability_find_testset', 'snr', 'snrmax', 'session_id', 'date', 'subject', 'SesIdx', 'AREALABEL', 'Area', 'waveform', 'spikepos_1', 'spikepos_2']


In [3]:
# 计算所有图像的response
# 参考 compute_neuron_responses.py，使用best_r_time1和best_r_time2确定的时间窗口
n_neurons = psth_data.shape[0]
n_stimuli = psth_data.shape[1]
n_time = psth_data.shape[2]

best_r_time1 = unit_info['best_r_time1'].values
best_r_time2 = unit_info['best_r_time2'].values

# 初始化响应矩阵
neuron_responses = np.zeros((n_neurons, n_stimuli), dtype=np.float32)

print(f"计算神经元响应 (使用最佳时间窗口)...")
print(f"时间窗口范围: [{best_r_time1.min()}, {best_r_time1.max()}] -> [{best_r_time2.min()}, {best_r_time2.max()}]")

for neuron_idx in range(n_neurons):
    start_time = int(best_r_time1[neuron_idx])
    end_time = int(best_r_time2[neuron_idx])
    
    # 处理时间窗口越界情况
    if start_time < 0:
        start_time = 0
    if end_time > n_time:
        end_time = n_time
    if end_time <= start_time:
        mid_time = n_time // 2
        start_time = max(0, mid_time - 5)
        end_time = min(n_time, mid_time + 5)
    
    # 计算该神经元在时间窗口内的平均响应
    psth_neuron = psth_data[neuron_idx, :n_stimuli, start_time:end_time]
    neuron_responses[neuron_idx] = np.mean(psth_neuron, axis=1)

print(f"神经元响应形状: {neuron_responses.shape}")

# 检查是否有NaN
nan_count = np.sum(np.isnan(neuron_responses))
print(f"NaN值数量: {nan_count}")

计算神经元响应 (使用最佳时间窗口)...
时间窗口范围: [20, 200] -> [90, 390]
神经元响应形状: (15652, 1072)
NaN值数量: 0


In [4]:
# 分割数据：前1000张和后72张
responses_1000 = neuron_responses[:, :1000]
responses_72 = neuron_responses[:, 1000:]


In [5]:
unit_info['Body_selectivity'] = (unit_info['B_SI'] > 0.3).astype(int)
unit_info['Face_selectivity'] = (unit_info['F_SI'] > 0.3).astype(int)
unit_info['Object_selectivity'] = (unit_info['O_SI'] > 0.3).astype(int)

n_selectivities = unit_info['Body_selectivity'] + unit_info['Face_selectivity'] + unit_info['Object_selectivity']
n_multi_selective = (n_selectivities >= 2).sum()
print(f"具有两个及以上选择性的神经元数量: {n_multi_selective} / {len(unit_info)} ({100*n_multi_selective/len(unit_info):.2f}%)")

具有两个及以上选择性的神经元数量: 684 / 15652 (4.37%)


In [10]:
import importlib.util
from pathlib import Path
from scipy.io import loadmat
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, confusion_matrix
from scipy.optimize import linear_sum_assignment

ROOT = Path('/media/ubuntu/sda/TrippleN')
CAPTIONS_PATH = ROOT / 'customize' / 'coco_captions_1000x5.pkl'
PREDICT_SCRIPT = ROOT / 'scripts' / 'predict_neuron_activity_gpu.py'
MODEL_KMEANS_DIR = ROOT / 'customize' / 'space_characteristic' / 'caption_kmeans_all_mpnet'
MODEL_KMEANS_DIR.mkdir(parents=True, exist_ok=True)

spec = importlib.util.spec_from_file_location('predict_neuron_activity_gpu', str(PREDICT_SCRIPT))
if spec is None or spec.loader is None:
    raise RuntimeError(f'无法加载脚本: {PREDICT_SCRIPT}')
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

device = mod.setup_device()
sentence_model = mod.load_sentence_model(device)
caption_features = mod.extract_caption_features(str(CAPTIONS_PATH), sentence_model)
caption_features = np.asarray(caption_features, dtype=np.float32)
print('caption_features shape:', caption_features.shape)

del sentence_model
mod.clear_gpu_memory()

cluster_info_by_k = {}
for k in range(8, 13):
    km = KMeans(n_clusters=k, random_state=0, n_init='auto')
    labels = km.fit_predict(caption_features)
    cluster_info_by_k[k] = labels.astype(np.int32)
    np.save(MODEL_KMEANS_DIR / f'cluster_idx_k{k}.npy', cluster_info_by_k[k])

cluster_info = cluster_info_by_k[12].reshape(1, -1)
print('cluster_info (k=12) shape for downstream:', cluster_info.shape)
print('saved dir:', MODEL_KMEANS_DIR)

使用设备: cuda


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4303.14it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: /media/ubuntu/sda/TrippleN/model/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


加载coco captions矩阵...
Caption矩阵形状: (1000, 5)
开始使用all-mpnet-base-v2提取 1000 张图片的caption特征...
  已处理 160/1000 张图片
  已处理 320/1000 张图片
  已处理 480/1000 张图片
  已处理 640/1000 张图片
  已处理 800/1000 张图片
  已处理 960/1000 张图片
Caption特征提取完成，特征矩阵形状: (1000, 768)
caption_features shape: (1000, 768)
cluster_info (k=12) shape for downstream: (1, 1000)
saved dir: /media/ubuntu/sda/TrippleN/customize/space_characteristic/caption_kmeans_all_mpnet


In [16]:
import re

def compute_si(rsp_cat, rsp_noncat, eps=1e-10):
    m_cat = np.mean(rsp_cat)
    m_non = np.mean(rsp_noncat)
    v_cat = np.var(rsp_cat)
    v_non = np.var(rsp_noncat)
    denom = np.sqrt(0.5 * (v_cat + v_non) + eps)
    return (m_cat - m_non) / denom

n_units = responses_1000.shape[0]
rmin = responses_1000.min(axis=1, keepdims=True)
rmax = responses_1000.max(axis=1, keepdims=True)
R = 2 * (responses_1000.astype(np.float64) - rmin) / (rmax - rmin + 1e-10) - 1

unit_info_by_k = {}
si_by_k = {}
cluster_labels_by_k = {}
summary_rows = []

for k in range(8, 13):
    cluster_flat = np.asarray(cluster_info_by_k[k]).flatten()
    cluster_labels_by_k[k] = cluster_flat.copy()
    cluster_ids = np.unique(cluster_flat)
    n_clusters = len(cluster_ids)

    SI_clusters = np.zeros((n_units, n_clusters))
    for j in range(n_clusters):
        mask_c = cluster_flat == cluster_ids[j]
        mask_n = ~mask_c
        m_cat = R[:, mask_c].mean(axis=1)
        m_non = R[:, mask_n].mean(axis=1)
        v_cat = R[:, mask_c].var(axis=1)
        v_non = R[:, mask_n].var(axis=1)
        SI_clusters[:, j] = (m_cat - m_non) / np.sqrt(0.5 * (v_cat + v_non) + 1e-10)

    unit_info_k = unit_info.copy()
    drop_cols = [c for c in unit_info_k.columns if re.match(r'^C\d+_(SI|selectivity)$', str(c))]
    if len(drop_cols) > 0:
        unit_info_k = unit_info_k.drop(columns=drop_cols)

    for j in range(n_clusters):
        unit_info_k[f'C{j}_SI'] = SI_clusters[:, j]
        unit_info_k[f'C{j}_selectivity'] = (SI_clusters[:, j] > 0.3).astype(int)

    n_sel = sum(unit_info_k[f'C{j}_selectivity'] for j in range(n_clusters))
    n_cluster_selective = int((n_sel >= 1).sum())
    unit_info_k['cluster_tuning_type'] = 'multi_tuning'
    unit_info_k.loc[n_sel == 0, 'cluster_tuning_type'] = 'broad_tuning'
    unit_info_k.loc[n_sel == 1, 'cluster_tuning_type'] = 'single_tuning'
    selective_cluster = np.full(len(unit_info_k), -1, dtype=int)
    for j in range(n_clusters):
        selective_cluster[(unit_info_k[f'C{j}_selectivity'] == 1) & (n_sel == 1)] = j
    unit_info_k['selective_cluster_idx'] = selective_cluster

    si_by_k[k] = SI_clusters
    unit_info_by_k[k] = unit_info_k

    summary_rows.append({'k': k, 'n_clusters': n_clusters, 'n_cluster_selective_units': n_cluster_selective})
    print(f'k={k}: C0~C{n_clusters-1} 已写入, 至少1个cluster选择性unit数 {n_cluster_selective}/{n_units}')

unit_info = unit_info_by_k[12]
cluster_info = cluster_labels_by_k[12].reshape(1, -1)

unit_info_all_k = unit_info.copy()
for k in range(8, 13):
    ui_k = unit_info_by_k[k]
    unit_info_all_k[f'cluster_tuning_type_k{k}'] = ui_k['cluster_tuning_type'].values
    unit_info_all_k[f'selective_cluster_idx_k{k}'] = ui_k['selective_cluster_idx'].values
    n_clusters_k = len(np.unique(cluster_labels_by_k[k]))
    n_sel_k = np.zeros(len(ui_k), dtype=int)
    for j in range(n_clusters_k):
        n_sel_k += ui_k[f'C{j}_selectivity'].values.astype(int)
    unit_info_all_k[f'n_cluster_selective_k{k}'] = n_sel_k

save_path_all_k = MODEL_KMEANS_DIR / 'unit_info_tuning_all_kmeans_k8to12.pkl'
unit_info_all_k.to_pickle(save_path_all_k)
print('saved:', save_path_all_k)

k=8: C0~C7 已写入, 至少1个cluster选择性unit数 11244/15652
k=9: C0~C8 已写入, 至少1个cluster选择性unit数 11598/15652
k=10: C0~C9 已写入, 至少1个cluster选择性unit数 11567/15652
k=11: C0~C10 已写入, 至少1个cluster选择性unit数 12073/15652
k=12: C0~C11 已写入, 至少1个cluster选择性unit数 12490/15652
saved: /media/ubuntu/sda/TrippleN/customize/space_characteristic/caption_kmeans_all_mpnet/unit_info_tuning_all_kmeans_k8to12.pkl


In [13]:
import seaborn as sns
from scipy.cluster import hierarchy
import os

if 'Body_selectivity' not in unit_info.columns:
    unit_info = unit_info.copy()
    unit_info['Body_selectivity'] = (unit_info['B_SI'] > 0.2).astype(int)
    unit_info['Face_selectivity'] = (unit_info['F_SI'] > 0.2).astype(int)
    unit_info['Object_selectivity'] = (unit_info['O_SI'] > 0.2).astype(int)

color_sel_none = 'lightgray'
color_sel_multi = '#EC6F7E'
color_sel_body = '#8AB07C'
color_sel_face = '#5E9FD1'
color_sel_object = '#EC6F7E'

FIG_DIR = '/media/ubuntu/sda/TrippleN/customize/figures'
os.makedirs(FIG_DIR, exist_ok=True)

saved_paths = []
for k in range(8, 13):
    unit_info_k = unit_info_by_k[k].copy()
    cluster_flat = np.asarray(cluster_labels_by_k[k]).flatten()
    cluster_ids = np.unique(cluster_flat)
    n_clusters = len(cluster_ids)

    B = unit_info_k['Body_selectivity'].values
    F = unit_info_k['Face_selectivity'].values
    O = unit_info_k['Object_selectivity'].values
    n_sel_bfo = B + F + O
    unit_color = np.array([color_sel_none] * len(unit_info_k))
    unit_color[(B == 1) & (F == 0) & (O == 0)] = color_sel_body
    unit_color[(F == 1) & (B == 0) & (O == 0)] = color_sel_face
    unit_color[(O == 1) & (B == 0) & (F == 0)] = color_sel_object
    unit_color[n_sel_bfo >= 2] = color_sel_multi

    R_raw = responses_1000.astype(np.float64)
    R_by_cluster = np.zeros((R_raw.shape[0], n_clusters))
    for j, cid in enumerate(cluster_ids):
        mask = cluster_flat == cid
        R_by_cluster[:, j] = R_raw[:, mask].mean(axis=1)

    r_min = R_by_cluster.min(axis=1, keepdims=True)
    r_max = R_by_cluster.max(axis=1, keepdims=True)
    range_r = r_max - r_min
    range_r[range_r == 0] = 1
    R_norm = 2 * (R_by_cluster - r_min) / range_r - 1

    cmap_cluster = plt.cm.tab20
    col_color = np.array([cmap_cluster(i % 20) for i in range(n_clusters)])
    col_color_hex = ['#%02x%02x%02x' % (int(r*255), int(g*255), int(b*255)) for r, g, b, _ in col_color]

    col_linkage = hierarchy.linkage(R_norm.T, method='average')
    col_order = hierarchy.leaves_list(col_linkage)
    col_order_inv = np.zeros(n_clusters, dtype=int)
    col_order_inv[col_order] = np.arange(n_clusters)

    selective_cluster = unit_info_k['selective_cluster_idx'].values.astype(int)
    tuning_type = unit_info_k['cluster_tuning_type'].values
    single_mask = tuning_type == 'single_tuning'
    multi_mask = tuning_type == 'multi_tuning'
    no_mask = tuning_type == 'broad_tuning'

    single_idx = np.where(single_mask)[0]
    if len(single_idx) > 0:
        single_sort = single_idx[np.argsort(col_order_inv[np.clip(selective_cluster[single_idx], 0, n_clusters-1)])]
    else:
        single_sort = np.array([], dtype=int)
    multi_idx = np.where(multi_mask)[0]
    no_idx = np.where(no_mask)[0]
    sort_idx = np.concatenate([single_sort, multi_idx, no_idx])

    R_norm_sorted = R_norm[sort_idx]
    unit_color_sorted = unit_color[sort_idx]
    tuning_type_ordered = tuning_type[sort_idx]
    sel_cluster_ordered = selective_cluster[sort_idx]

    row_color_by_tuning = np.array(['#e0e0e0'] * len(sort_idx))
    row_color_by_tuning[tuning_type_ordered == 'multi_tuning'] = '#EC6F7E'
    for j in range(n_clusters):
        row_color_by_tuning[(tuning_type_ordered == 'single_tuning') & (sel_cluster_ordered == j)] = col_color_hex[j]

    n_plot_units = None
    if n_plot_units is not None and R_norm_sorted.shape[0] > n_plot_units:
        sub_idx = np.arange(min(n_plot_units, R_norm_sorted.shape[0]))
        R_plot = R_norm_sorted[sub_idx]
        row_colors = [np.asarray(row_color_by_tuning[sub_idx]), np.asarray(unit_color_sorted[sub_idx])]
    else:
        R_plot = R_norm_sorted
        row_colors = [np.asarray(row_color_by_tuning), np.asarray(unit_color_sorted)]

    col_colors = [np.asarray(col_color_hex)]
    g = sns.clustermap(
        R_plot,
        row_cluster=False,
        col_cluster=True,
        method='average',
        cmap='RdBu_r',
        center=0,
        vmin=-1,
        vmax=1,
        figsize=(14, 10),
        dendrogram_ratio=(0.15, 0.1),
        row_colors=row_colors,
        col_colors=col_colors,
    )
    g.ax_heatmap.set_xlabel('Cluster')
    g.ax_heatmap.set_ylabel('Unit')
    plt.suptitle(f'k={k} | Rows: single (by col cluster order) | multi | no', y=1.02)

    out_pdf = os.path.join(FIG_DIR, f'characterization_1000nsd_heatmap_k{k}.pdf')
    g.fig.savefig(out_pdf, bbox_inches='tight')
    plt.close(g.fig)
    saved_paths.append(out_pdf)

print('saved heatmaps:')
for p in saved_paths:
    print(p)

saved heatmaps:
/media/ubuntu/sda/TrippleN/customize/figures/characterization_1000nsd_heatmap_k8.pdf
/media/ubuntu/sda/TrippleN/customize/figures/characterization_1000nsd_heatmap_k9.pdf
/media/ubuntu/sda/TrippleN/customize/figures/characterization_1000nsd_heatmap_k10.pdf
/media/ubuntu/sda/TrippleN/customize/figures/characterization_1000nsd_heatmap_k11.pdf
/media/ubuntu/sda/TrippleN/customize/figures/characterization_1000nsd_heatmap_k12.pdf


In [18]:
from itertools import combinations
from sklearn.metrics import confusion_matrix
from matplotlib.backends.backend_pdf import PdfPages

MODEL_KMEANS_DIR = Path('/media/ubuntu/sda/TrippleN/customize/space_characteristic/caption_kmeans_all_mpnet')
unit_info_all_k = pd.read_pickle(MODEL_KMEANS_DIR / 'unit_info_tuning_all_kmeans_k8to12.pkl')

k_list = [8, 9, 10, 11, 12]
label_order = ['broad_tuning', 'single_tuning', 'multi_tuning']

out_pdf = MODEL_KMEANS_DIR / 'tuning_confusion_matrix_k8to12_pairs.pdf'
out_csv = MODEL_KMEANS_DIR / 'tuning_confusion_matrix_k8to12_pairs_counts.csv'

rows = []
with PdfPages(out_pdf) as pdf:
    for k_ref, k_cmp in combinations(k_list, 2):
        y_ref = unit_info_all_k[f'cluster_tuning_type_k{k_ref}'].astype(str).values
        y_cmp = unit_info_all_k[f'cluster_tuning_type_k{k_cmp}'].astype(str).values

        cm = confusion_matrix(y_ref, y_cmp, labels=label_order)
        cm_df = pd.DataFrame(cm, index=label_order, columns=label_order)

        cm_row = cm.astype(float)
        row_sum = cm_row.sum(axis=1, keepdims=True)
        row_sum[row_sum == 0] = 1
        cm_row = cm_row / row_sum
        cm_row_df = pd.DataFrame(cm_row, index=label_order, columns=label_order)

        for i, rlab in enumerate(label_order):
            for j, clab in enumerate(label_order):
                rows.append(
                    {
                        'k_ref': k_ref,
                        'k_cmp': k_cmp,
                        'ref_label': rlab,
                        'cmp_label': clab,
                        'count': int(cm[i, j]),
                        'row_norm': float(cm_row[i, j]),
                    }
                )

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
        sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0])
        axes[0].set_title(f'count | k{k_ref} (row) vs k{k_cmp} (col)')
        axes[0].set_xlabel(f'k{k_cmp}')
        axes[0].set_ylabel(f'k{k_ref}')

        sns.heatmap(cm_row_df, annot=True, fmt='.2f', cmap='OrRd', vmin=0, vmax=1, cbar=False, ax=axes[1])
        axes[1].set_title('row-normalized proportion')
        axes[1].set_xlabel(f'k{k_cmp}')
        axes[1].set_ylabel(f'k{k_ref}')

        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

cm_all_df = pd.DataFrame(rows)
cm_all_df.to_csv(out_csv, index=False)

print('saved:', out_pdf)
print('saved:', out_csv)
cm_all_df.head()

saved: /media/ubuntu/sda/TrippleN/customize/space_characteristic/caption_kmeans_all_mpnet/tuning_confusion_matrix_k8to12_pairs.pdf
saved: /media/ubuntu/sda/TrippleN/customize/space_characteristic/caption_kmeans_all_mpnet/tuning_confusion_matrix_k8to12_pairs_counts.csv


,k_ref,k_cmp,ref_label,cmp_label,count,row_norm
0,8,9,broad_tuning,broad_tuning,3829,0.868648
1,8,9,broad_tuning,single_tuning,553,0.125454
2,8,9,broad_tuning,multi_tuning,26,0.005898
3,8,9,single_tuning,broad_tuning,221,0.035315
4,8,9,single_tuning,single_tuning,5376,0.859060
